# Music Synthesis

In the second part of the Music Synthesis Lab, we will improve the rendering of the piece of music that we started last week.

* Recall that the piece contained three voices, but so far we have considered only the first of these voices. The first improvement will be to combine all **three voices**.
* In a second step, we will replace the pure tones we synthesized last week with tones and a few **harmonics**. This should make the synthesized sound more natural and richer.
* Finally, we modify the abrupt switching on and off of tones with a more gradual and natural fading in and out of notes using **envelopes**. This also should make the sound mare natural and piano-like.

Refer to the notebook for first part of the lab if you need to remind yourself of the underlying concepts.

**Note:** In contrast to prior labs, this lab is less is slightly structured. Additionally, you are expected to experiment with these improvements to synthesize a rendering of the music that sounds as good as you can make it.

In [ ]:
# Import the usual suspects, including the Audio widget
%matplotlib inline
import matplotlib.pyplot as plt

import numpy as np

from IPython.display import Audio

## This notebook is incomplete

In this notebook, there are multiple places for you to fill in either code or text.

You should do that directly in this notebook. 

Once you have completed all your work in this notebook, rerun the entire notebook using "Kernel > Restart and Run All" from the menubar. 

Fix any errors, then remove this cell (your notebook is now complete), and submit a PDF version.

## Code from the First Part

We wrote several functions last week that we wil re-use or improve upon. The functions we need are repeated below so that we have them at our disposal.

Specifically, we need the following functions:
* `key_to_freq` - translates a key number to the corresponding (fundamental) frequency
* `key_to_tone` - synthesizes a tone for a given key number and duration
* `synthesize` - synthesizes a complete piece from given sequences of key numbers, start times, and durations.

In [ ]:
def key_to_freq( k ):
    """Compute the frequency that corresponds to piano key k
    
    Parameters:
    -----------
    k - key number

    Returns:
    --------
    frequency of tone
    """
    e = (k - 49) / 12

    return 440 * (2 ** e) 

In [ ]:
def key_to_tone(k, dur, fs=11025):
    """Synthesize a tone of given frequency and duration
    
    Parameters:
    -----------
    k - piano key number
    dur - duration in seconds
    fs - sample rate

    Returns:
    --------
    NumPy vector of samples
    """
    tt = np.arange(0, dur, 1/fs)
    freq = key_to_freq(k)

    return np.cos(2*np.pi * freq * tt)

In [ ]:
def synthesize(keys, durations, startPulses, BPM, fs=11025, PPQ=4):
    """Synthesize a sequence of tones
    
    Parameters:
    -----------
    keys - a vector that specifies the piano keys and, thus, the frequencies to be played
    durations - a vector that specifies the length of each note in pulses
    startPulses - a vector that provides the start time of each tone in pulses
    BPM - tempo variable 
    fs - sample rate with a default value of 11025
    PPQ - the number of pulses per beat with a default value of 4.

    Returns:
    --------
    a vector of samples
    """

    # derived timing parameters
    beats_per_second = BPM / 60
    seconds_per_beat = 1 / beats_per_second    # also duration of a quarter note
    seconds_per_pulse = seconds_per_beat / PPQ # duration of a pulse

    samples_per_pulse = int( seconds_per_pulse * fs )

    # Allocate memory to hold the signal
    Ns = startPulses[-1] * samples_per_pulse + len(key_to_tone(keys[-1], durations[-1]*seconds_per_pulse, fs)) 
    xx = np.zeros(Ns)

    # synthesize
    for n in range(len(keys)):
        # tone has duration `durations[n]*seconds_per_pulse``
        tone = key_to_tone(keys[n], durations[n] * seconds_per_pulse, fs) 
        
        # first sample for this pulse
        seg = startPulses[n] * samples_per_pulse
        
        # length of tone in samples
        Nt = len(tone)

        # insert tone
        xx[seg : seg+Nt] = tone

    return xx

## Loading Data

Like in the first part, we load the data file for the musice. The data are stored in  `pickle` format. The tools to read a file in this format are part of the Python standard library.

The code below loads the data that represent the sheet music into the variable `the_voices`.

In [ ]:
## load data from pickle file
# load the pickle module
import pickle

# read data from file
with open('bach_fugue.pkl', 'rb') as f:
    # The protocol version used is detected automatically, so we do not
    # have to specify it.
    the_voices = pickle.load(f)

Recall that `the_voices` is a Python list with three elements - one for each of the three voices.

Each element of the list, e.g., `the_voices[0]`, contains a Python dictionary with the following keys:
* `startPulses` is the key for an array that indicates the start times (in pulses) for notes to be played
* `durations` is the key for an array that indicates the note durations (in pulses)
* `noteNumbers` is the key for the notes to be played (expressed a piano key numbers)

Note that the three sequences stored in each of the voices are precisely the inputs to the `synthesize` function. Thus, we can easily synthesize one of the voices as in the cell below.

<p style="margin-bottom: 15px; padding: 4px 12px; background-color: #e7f3fe; border-left: 6px solid #2196F3;">
Executing the cell below also verifies that all existing code code is working properly.
</p>


In [ ]:
## synthesize music
# which voice to play
n_voice = 0

# synthesize
this_voice = the_voices[n_voice]
ss = synthesize(this_voice['noteNumbers'],
                this_voice['durations'],
                this_voice['startPulses'],
                BPM=80,
                fs=11025,
                PPQ=4)

# play it
Audio(ss, rate=11025)

This is as far as we got last week. Let's start improving.

## Task: Combine Three Voices

Our first task is combine the three voices and play them simultaneously.

Towards that end, write a function `combine_voices` that produces a signal that is equal to the sum of the three voices. The function takes the following inputs:
* `the_voices` - is a list of dictionaries; the dictionaries store sequences for key numbers, start pulses, and durations under the keys `noteNumbers`, `startPulses`, and `durations`
* `BPM` - beats-per-minute
* `fs` - sample rate
* `PPQ` - the number of pulses per quarter-note; this parameter defaults to 4

The function returns a NumPy array that holds the samples for the synthesized piece.

Your function should call the `synthesize` function for each voice (as in the last code cell above) and then add the voices.

<p style="margin-bottom: 15px; padding: 4px 12px; background-color: #ffdddd; border-left: 6px solid #f44336;">
When adding the signals for the individual voices, you must be careful since the signals for the voices are not all of the same length. Use appropriate slice syntax. You may also need to determine the length of the longest voice when you allocate the array that wil hold the samples.
</p>


In [ ]:
def combine_voices(the_voices, BPM, fs, PPQ=4):
    """ synthesize the signal for multiple voices
    
    Paramters
    ---------
    the_voices - a list of dictionaries, each holding information to synthesize one voice
    BPM - beats-per-minute
    fs - sample rate
    PPQ - pulses per quarter note (default value: 4)

    Returns
    -------
    signal containing combination of all voices
    """

    FILL_ME_IN

    return sig


Test your function using the cell below. You should be able to clearly hear the more complex sound from the simultaneous voices.

In [ ]:
ss = combine_voices(the_voices, 80, 11025, 4)

Audio(ss, rate=11025)

## Task: Add Harmonics

When a musical instrument plays a tone, it doesn't produce a single frequency tone, For example, when a string in a piano is struck the string vibrates mainly at the fundamental frequency (the frequency that corresponds to the key number). 

However, there will also be smaller vibrations at integer multiples of the fundamental frequency. These frequencies are called *harmonic frequencies* or just *harmonics*.

Your second task is to modify the function `key_to_tone` such that it produces the sum of tones at the fundamental frequency and several harmonics. To facilitate this additions we add the parameter `harmonics` to the function.
* this parameter has a default value of `None`
* if the value of `harmonics == None` then only the fundamental frequency is played (as before)
* to specify harmonics, the parameter `harmonics` contains a list (or NumPy array) of amplitudes for the harmonics
  - for example if `harmonics` has three elements, then three harmonic frequencies are synthesized in addition to the fundamental frequency
  - the elements of `harmonics` specify the amplitude of each harmonic
  - these amplitudes should be smallerr than the amplitude of the fundamental tone (i.e., less than one)

<p style="margin-bottom: 15px; padding: 4px 12px; background-color: #ffffcc; border-left: 6px solid #ffeb3b;">
To avoid problems with undersampling, the largest harmonic frequency must be less than half the sampling rate. You should either keep the number of harmonics small enough to avoid this issue or increase the sample rate.
</p>

The revised function `key_to_tone` is defined as follows:

In [ ]:
def key_to_tone(k, dur, fs=11025, harmonics=None):
    """Synthesize a tone of given frequency and duration
    
    Parameters:
    -----------
    k - piano key number
    dur - duration in seconds
    fs - sample rate
    harmonics - list of amplitudes for harmonics (defaul value: None; no  harmonics synthesized)

    Returns:
    --------
    NumPy vector of samples
    """
    tt = np.arange(0, dur, 1/fs)
    
    FILL_ME_IN

    return sig

### Spectrogram

To verify that your function works correctly, compute the spectrogram.

Since `harmonics` has three elements, we should see four lines in the spectrogram - one for the fundamental frequency and three for the harmonics. 

The lines in the spectrogram should be equally spaced since the frequencies are integer multiples of the fundamental frequency.

In [ ]:
harmonics = [0.5, 0.25, 0.125]
ss = key_to_tone(49, 1, 11025, harmonics)

plt.specgram(ss, Fs=11025, NFFT=1024)
plt.show()

### Integrating the revised `key_to_tone`

To make use of the new `key_to_tone` function, we must update the `synthesize` and `combine_voices` functions.

However, these changes are minimal:
* for both functions, we need to add a parameter `harmonics` so that we can specify the harmonics that are passed to `key_to_tone`
* in the `synthesize` function, update the call to `key_to_freq` to include the parameter `harmonics`
* in the `combine_voices` function, update the call to `synthesize` to include the parameter `harmonics`

Update the functions in the cells below to incorporate the parameter `harmonics`.

In [ ]:
def synthesize(keys, durations, startPulses, BPM, fs=11025, harmonics=None, PPQ=4):
    """Synthesize a sequence of tones
    
    Parameters:
    -----------
    keys - a vector that specifies the piano keys and, thus, the frequencies to be played
    durations - a vector that specifies the length of each note in pulses
    startPulses - a vector that provides the start time of each tone in pulses
    BPM - tempo variable 
    fs - sample rate with a default value of 11025
    harmonics - an iterable containing the amplitudes for the harmonic frequencies (default: `None`, no harmonics)
    PPQ - the number of pulses per beat with a default value of 4.

    Returns:
    --------
    a vector of samples
    """

    # derived timing parameters
    beats_per_second = BPM / 60
    seconds_per_beat = 1 / beats_per_second    # also duration of a quarter note
    seconds_per_pulse = seconds_per_beat / PPQ # duration of a pulse

    samples_per_pulse = int( seconds_per_pulse * fs )

    # Allocate memory to hold the signal
    Ns = startPulses[-1] * samples_per_pulse + len(key_to_tone(keys[-1], durations[-1]*seconds_per_pulse, fs)) 
    xx = np.zeros(Ns)

    # synthesize
    for n in range(len(keys)):
        # tone has duration `durations[n]*seconds_per_pulse``
        tone = key_to_tone(keys[n], durations[n] * seconds_per_pulse, fs)    # <========= FIX ME 
        
        # first sample for this pulse
        seg = startPulses[n] * samples_per_pulse
        
        # length of tone in samples
        Nt = len(tone)

        # insert tone
        xx[seg : seg+Nt] = tone

    return xx

In [ ]:
def combine_voices(the_voices, BPM, fs, harmonics=None, PPQ=4):
    """ synthesize the signal for multiple voices
    
    Paramters
    ---------
    the_voices - a list of dictionaries, each holding information to synthesize one voice
    BPM - beats-per-minute
    fs - sample rate
    harmonics - an iterable containing the amplitudes for the harmonic frequencies (default: `None`, no harmonics)
    PPQ - pulses per quarter note (default value: 4)

    Returns
    -------
    signal containing combination of all voices
    """

    # figure out which voices is the longest
    len_voices = np.array([v['startPulses'][-1] + v['durations'][-1] for v in the_voices])
    ind_max = np.argmax(len_voices)

    # synthesize the longest voice first
    v = the_voices[ind_max]
    sig = synthesize(v['noteNumbers'], v['durations'], v['startPulses'], BPM, fs, PPQ)  # <============== FIX ME

    # now add the other voices, carefully
    for n in range(len(the_voices)):
        if n == ind_max:
            continue

        v = the_voices[n]
        xx = synthesize(v['noteNumbers'], v['durations'], v['startPulses'], BPM, fs, PPQ) # <============== FIX ME

        sig[:len(xx)] = sig[:len(xx)] + xx

    return sig

### Synthesis with Harmonics

The cell below synthesizes the piece with ten harmonics.

As this will ceate tones with frequencies approaching 10KHz, we must switch to a higher sampling rate to avoid undersampling.

In [ ]:
# amplitude of the harmonics decays gradually with frequency
harmonics = 1/(2*np.arange(1, 10)+1)

ss = combine_voices(the_voices, 80, 11025*2, harmonics, 4)

Audio(ss, rate=11025*2)

#### Spectrogram

To conclude this task, we plot the spectrogram for the entire piece with 10 harmonics.

The spectrogram shows a much richer frequency content than before. In addition to the fundamental frequency, the harmonic frequencies generate spectral contents at frequencies up to approximately 10KHz.

In [ ]:
plt.specgram(ss + 1e-5*np.random.randn(len(ss)), NFFT=1024, Fs=22050)
plt.colorbar()
plt.show()

## Task: ADSR Envelope

The synthesized sound sounds more complex with the added harmonics. However, it is still very artificial and the clicking sounds from the abrupt transition between tones is particularly annoying.

To make the transition between notes smoother and the synthesized music more natural sounding, we will add an *envelope* to each tone. An envelope captures how the amplitude of a note varies over the duration of each note.

As shown in the figure below below, the envelope has four phases: Attack, Delay, Sustain, and Release. Based on the initial letters of these phases, the envelope is known as an ADSR envelope.

An envelope multiplies each note. Therefore, an envelope must have the same duration as the note it multiplies. This also implies that tones of different duration must have different envelopes.

To obtain a uniform description of the envelope signal $e(t)$ we specify it relative to the duration of the note $T_n$:
* **A** The attack phase starts at the start of the note ($t=0$) and lasts $T_A$ percent of the duration $T_n$ of the note. During the attack phase, the envelope increases linearly from 0 to 1.
* **D** The decay phase follows immediately after the attack; it lasts $T_A$ percent of the note duration. During this time, the envelope decreases linearly from 1 to the sustain level $S$.
* **S** During the sustain phase, the envelope remains constant at level $S$. The sustain phases follows the decay phase and preceded the release. The duration of the sustain phase is inferred from the note duration and the durations of the other three phases.
* **R** The final phase is the release; it lasts $T_R$ percent of the note duration. During that time, the envelope decays linearly from the sustain level $S$ to 0.

![ADSR Envelope](ADSR_envelope.png)

Based on the above discussions, the envelope $e(t)$ for a note of duration $T_n$ can be described as follows. Note that the envelope is written in terms of $t/T_n$ to make all times relative to the note duration $T_n$.
$$
e(t) = \begin{cases}
\frac{1}{T_A} \frac{t}{T_n} & \text{for $0 \leq \frac{t}{T_n} < T_A$}\\
1 + (1-S) \frac{T_A}{T_D} - \frac{1-S}{T_D} \frac{t}{T_n}  & \text{for $T_A \leq \frac{t}{T_n} < T_A+T_D$} \\
S & \text{for $T_A + T_D \leq \frac{t}{T_n} < 1-T_R$}\\
\frac{S}{T_R} - \frac{S}{T_R} \frac{t}{T_n} & \text{for $1-T_R \leq \frac{t}{T_n} \leq 1$}
\end{cases}
$$

The envelope consists of four straight line segments that depend on the four envelope parameters attack duration $T_A$, decay duration $T_D$, sustain level $S$, and release duration $T_R$.

Your task is to write a function `envelope` to construct the envelope signal for a given duration and sampling rate. The function accepts the ADSR parameters as a dictionary with keys `A`, `D`, `S`, and `R` so that the signature of the function is:
``` python
def envelope(dur, fs, ADSR_dict):
    """construct envelope signal"""
```

The function must return a NumPy array.

Recall that the durations of the various phases are specified as a percentage of the note duration. Hence, all four parameters should be between 0 and 1 and the sum of $T_A$, $T_D$, and $T_R$ must be at most equal to 1.

**Hint:** it is probably best to start by constructing exactly the same time grid as that used by the `key_to_tone` function, i.e., with `tt = np.arange(0, dur, 1/fs)`. This ensure that the tone and the envelope will have the same length and can be easily multiplied.

**Suggestion:** consider logical indexing to construct the four linear segments.

In [ ]:
def envelope(dur, fs, ADSR_dict):
    """construct envelope signal
    
    Parameters:
    -----------
    dur - note duration in seconds
    fs - sample rate
    ADSR_dict - dictionary with ADSR parameters stored under keys `A`, `D`, `S`, and `R`

    Returns:
    --------
    Numpy array with samples of the envelope signal
    """
    # Initialization
    tt = np.arange(0, dur, 1/fs)

    # attack phase
    FILL_ME_IN

    # decay phase
    FILL_ME_IN

    # sustain phase
    FILL_ME_IN
    
    # release phase
    FILL_ME_IN

    return e


To verify that the envelope is constructed correctly, we synthesize an envelope and plot it as a function of time. 

In [ ]:
# ADSR parameters: short attack and release, long gradual decay
ADSR = {'A': 0.05,
        'D': 0.8,
        'S': 0.1,
        'R': 0.05}

# envelope
env = envelope(2, 1000, ADSR)

# plot
plt.plot(np.arange(0, 2, 1/1000), env)
plt.xlabel('Time (s)')
plt.ylabel('$e(t)$')
plt.grid()
plt.show()

### Single Note with Envelope

To assess the effect of the envelope on the sound of a note, we synthesize a note and multiply it with the envelope. Then, we play the resulting signal - the difference is quite dramatic.

In [ ]:
## parameters for tone
ADSR = {'A': 0.05,
        'D': 0.8,
        'S': 0.1,
        'R': 0.05}
harmonics = [0.5, 0.25, 0.125]
dur = 0.25
fs = 11025

env = envelope(dur, fs, ADSR)
ss = key_to_tone(49, dur, fs, harmonics) * env

Audio(ss, rate=fs)

### Putting it all Together

Now, all that is left is to modify the functions `synthesize` and `combine_voices` to incorporate the shaping of tones using envelopes.

We must pass the dictionary that the `envelope` function needs to both functions.

Additionally, the `synthesize` function must multiply the synthesized tones with the envelope. 

In [ ]:
def synthesize(keys, durations, startPulses, BPM, fs=11025, harmonics=None, PPQ=4, ADSR_dict=None):
    """Synthesize a sequence of tones
    
    Parameters:
    -----------
    keys - a vector that specifies the piano keys and, thus, the frequencies to be played
    durations - a vector that specifies the length of each note in pulses
    startPulses - a vector that provides the start time of each tone in pulses
    BPM - tempo variable 
    fs - sample rate with a default value of 11025
    harmonics - an iterable containing the amplitudes for the harmonic frequencies (default: `None`, no harmonics)
    PPQ - the number of pulses per beat with a default value of 4.
    ADSR_dict - dictionary for specifying ADSR envelope (Default: `None`, no shaping)

    Returns:
    --------
    a vector of samples
    """

    # derived timing parameters
    beats_per_second = BPM / 60
    seconds_per_beat = 1 / beats_per_second    # also duration of a quarter note
    seconds_per_pulse = seconds_per_beat / PPQ # duration of a pulse

    samples_per_pulse = int( seconds_per_pulse * fs )

    # Allocate memory to hold the signal
    Ns = startPulses[-1] * samples_per_pulse + len(key_to_tone(keys[-1], durations[-1]*seconds_per_pulse, fs)) 
    xx = np.zeros(Ns)

    # synthesize
    for n in range(len(keys)):
        # tone has duration `durations[n]*seconds_per_pulse``
        tone = key_to_tone(keys[n], durations[n] * seconds_per_pulse, fs, harmonics) 
        
        # first sample for this pulse
        seg = startPulses[n] * samples_per_pulse
        
        # length of tone in samples
        Nt = len(tone)

        # shaping with envelope
        if ADSR_dict is not None:
            tone = tone # <========= FIX ME: multiply with envelope

        # insert tone
        xx[seg : seg+Nt] = tone

    return xx

In [ ]:
def combine_voices(the_voices, BPM, fs, harmonics=None, PPQ=4, ADSR_dict=None):
    """ synthesize the signal for multiple voices
    
    Paramters
    ---------
    the_voices - a list of dictionaries, each holding information to synthesize one voice
    BPM - beats-per-minute
    fs - sample rate
    harmonics - an iterable containing the amplitudes for the harmonic frequencies (default: `None`, no harmonics)
    PPQ - pulses per quarter note (default value: 4)

    Returns
    -------
    signal containing combination of all voices
    """

    # figure out which voices is the longest
    len_voices = np.array([v['startPulses'][-1] + v['durations'][-1] for v in the_voices])
    ind_max = np.argmax(len_voices)

    # synthesize the longest voice first
    v = the_voices[ind_max]
    sig = synthesize(v['noteNumbers'], v['durations'], v['startPulses'], BPM, fs, harmonics, PPQ) # <======== FIX ME: ADSR_dict

    # now add the other voices, carefully
    for n in range(len(the_voices)):
        if n == ind_max:
            continue

        v = the_voices[n]
        xx = synthesize(v['noteNumbers'], v['durations'], v['startPulses'], BPM, fs, harmonics, PPQ) # <======== FIX ME: ADSR_dict

        sig[:len(xx)] = sig[:len(xx)] + xx

    return sig

With these two functions in place, we synthesize the entire piece again.

In [ ]:
# specify harmonics and ADSR envelope
harmonics = 1/(2*np.arange(1, 10)+1)
ADSR = {'A': 0.05,
        'D': 0.8,
        'S': 0.1,
        'R': 0.05}

# synthesize with these parameters
ss = combine_voices(the_voices, 80, 11025*2, harmonics, 4, ADSR)

Audio(ss, rate=11025*2)

## Fine-tuning

We now have two sets of parameters, the number and amplitudes of the harmonics and the shape of the envelope signal, to tweak the synthesized sound.

Experiment with these parameters and try to make the synthesized music sound as good as you can. Then, document your choices by:

* describing the values of all parameters you chose
* plotting the envelope function
* plotting the spectrogram of a single note
* a discussion 
  - which qualities of the synthesized sound are good, and
  - which aspects can still be improved
* a discussion of ideas for possible further improvements

Add additional cells (code and text) below as needed.

## Summary

We made significant improvements to the synthesized music and made an effort to fine-tune the quality of the synthesized sound.

### Deliverable

Submit a PDF version of this notebook. Make sure that:
* all cells are properly typeset and the "Incomplete" cell near the top is removed
* all code cells are complete (no `FILL_ME_IN` left)
* all functions are properly documented
* all plots have proper axis labels and other adornments as appropriate
* the section "Fine-tuning" contains required results and discussions.